In [51]:
# Install the libraries (only needed once)
# !pip install requests beautifulsoup4
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

def scrape_books(min_rating, max_price):
    pass

In [53]:
# 1- Pick the URL and save it
catalogue_url = "https://books.toscrape.com/"

# 2- Identify ourselves as a browser (tested and working!) = HEADERS
headers = {"User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/116.0.0.0 Safari/537.36"}
# "I am Chrome 116, running on Windows 10 64-bit, using WebKit as my rendering engine."

# 3- Make the request, passing the URL and the HEADERS
response = requests.get(catalogue_url, headers=headers)

# 4- Check the status (Good Practice!)
if response.status_code == 200:
    print("Connection successful!")
else:
    print(f"Connection failed: {response.status_code}")

Connection successful!


In [78]:
#Parcing the response
soup = BeautifulSoup(response.content, "html.parser")


In [57]:
# Find every book card on the first catalogue page
book_cards = soup.select("article.product_pod")

In [58]:
print(len(book_cards))

20


In [60]:
first_book = book_cards[0]

In [61]:
title = first_book.select_one("h3 a")["title"]
price = first_book.select_one("p.price_color").get_text(strip=True)
rating_word = first_book.select_one("p.star-rating")["class"][1]

print(title)
print (price)
print (rating_word)

A Light in the Attic
£51.77
Three


In [62]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

rating = rating_map[rating_word]

print(rating)

3


In [63]:
price_number = float(price.replace("£", ""))

print(price_number)

51.77


In [65]:
min_rating = 4.0
max_price = 20
books_data = []

for book in book_cards:
    title = book.select_one("h3 a")["title"]
    
    price_text = book.select_one("p.price_color").get_text(strip=True)
    price_number = float(price_text.replace("£", ""))
    
    rating_word = book.select_one("p.star-rating")["class"][1]
    rating = rating_map[rating_word]

    # Filtering condition
    if rating >= min_rating and price_number <= max_price:
        books_data.append({
            "Title": title,
            "Price (£)": price_number,
            "Rating": rating
        })

In [66]:
print(len(books_data))

1


In [67]:
print(books_data)

[{'Title': 'Set Me Free', 'Price (£)': 17.46, 'Rating': 5}]


In [ ]:
print(books_data[0])

In [79]:
current_url = catalogue_url

response = requests.get(catalogue_url, headers=headers)
soup = BeautifulSoup(response.content, "html.parser")

next_button = soup.select_one("li.next a")
next_page = next_button["href"]
print(next_page)

catalogue/page-2.html


In [80]:
next_url = urljoin(current_url, next_page)
print (next_url)

https://books.toscrape.com/catalogue/page-2.html


In [81]:
for book in book_cards:

    title = book.select_one("h3 a")["title"]

    # Get the book detail page link
    book_link = book.select_one("h3 a")["href"]

    print(title)
    print(book_link)
    print("----------------")

A Light in the Attic
catalogue/a-light-in-the-attic_1000/index.html
----------------
Tipping the Velvet
catalogue/tipping-the-velvet_999/index.html
----------------
Soumission
catalogue/soumission_998/index.html
----------------
Sharp Objects
catalogue/sharp-objects_997/index.html
----------------
Sapiens: A Brief History of Humankind
catalogue/sapiens-a-brief-history-of-humankind_996/index.html
----------------
The Requiem Red
catalogue/the-requiem-red_995/index.html
----------------
The Dirty Little Secrets of Getting Your Dream Job
catalogue/the-dirty-little-secrets-of-getting-your-dream-job_994/index.html
----------------
The Coming Woman: A Novel Based on the Life of the Infamous Feminist, Victoria Woodhull
catalogue/the-coming-woman-a-novel-based-on-the-life-of-the-infamous-feminist-victoria-woodhull_993/index.html
----------------
The Boys in the Boat: Nine Americans and Their Epic Quest for Gold at the 1936 Berlin Olympics
catalogue/the-boys-in-the-boat-nine-americans-and-their

In [82]:
book_url = urljoin(catalogue_url, book_link)

print(book_url)

https://books.toscrape.com/catalogue/its-only-the-himalayas_981/index.html


In [83]:
detail_response = requests.get(book_url, headers=headers)

detail_soup = BeautifulSoup(
    detail_response.content,
    "html.parser"
)

print(detail_soup.prettify()[:2000])

<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!-->
<html class="no-js" lang="en-us">
 <!--<![endif]-->
 <head>
  <title>
   It's Only the Himalayas | Books to Scrape - Sandbox
  </title>
  <meta content="text/html; charset=utf-8" http-equiv="content-type"/>
  <meta content="24th Jun 2016 09:29" name="created"/>
  <meta content="
    “Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a few things her mother might classify as &quot;stupid.&quot; She swam with great white sharks in South Africa, ran from lions in Zimbabwe, climbed a Himalayan mountain without training in Nepal, and wa “Wherever you go, whatever you do, just

In [84]:
product_table = detail_soup.select_one("table.table-striped")

print(product_table.prettify())

<table class="table table-striped">
 <tr>
  <th>
   UPC
  </th>
  <td>
   a22124811bfa8350
  </td>
 </tr>
 <tr>
  <th>
   Product Type
  </th>
  <td>
   Books
  </td>
 </tr>
 <tr>
  <th>
   Price (excl. tax)
  </th>
  <td>
   £45.17
  </td>
 </tr>
 <tr>
  <th>
   Price (incl. tax)
  </th>
  <td>
   £45.17
  </td>
 </tr>
 <tr>
  <th>
   Tax
  </th>
  <td>
   £0.00
  </td>
 </tr>
 <tr>
  <th>
   Availability
  </th>
  <td>
   In stock (19 available)
  </td>
 </tr>
 <tr>
  <th>
   Number of reviews
  </th>
  <td>
   0
  </td>
 </tr>
</table>



In [88]:
for row in product_table.select("tr"):
    header = row.select_one("th").get_text(strip=True)
    
    if header == "UPC":
        upc = row.select_one("td").get_text(strip=True)

print(upc)

a22124811bfa8350


In [89]:
availability = None

for row in product_table.select("tr"):
    header = row.select_one("th").get_text(strip=True)

    if header == "Availability":
        availability = row.select_one("td").get_text(strip=True)

print(availability)

In stock (19 available)


In [90]:
breadcrumb = detail_soup.select_one("ul.breadcrumb")

print(breadcrumb.prettify())

<ul class="breadcrumb">
 <li>
  <a href="../../index.html">
   Home
  </a>
 </li>
 <li>
  <a href="../category/books_1/index.html">
   Books
  </a>
 </li>
 <li>
  <a href="../category/books/travel_2/index.html">
   Travel
  </a>
 </li>
 <li class="active">
  It's Only the Himalayas
 </li>
</ul>



In [91]:
genres = detail_soup.select("ul.breadcrumb li a")

for genre in genres:
    print(genre.get_text(strip=True))

Home
Books
Travel


In [93]:
genre = detail_soup.select(
    "ul.breadcrumb li a"
)[2].get_text(strip=True)

print(genre)

Travel


In [94]:
description_header = detail_soup.select_one("#product_description")

print(description_header)

<div class="sub-header" id="product_description">
<h2>Product Description</h2>
</div>


In [95]:
description = description_header.find_next("p").get_text(strip=True)

print(description)

“Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a few things her mother might classify as "stupid." She swam with great white sharks in South Africa, ran from lions in Zimbabwe, climbed a Himalayan mountain without training in Nepal, and wa “Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a few things her mother might classify as "stupid." She swam with great white sharks in South Africa, ran from lions in Zimbabwe, climbed a Himalayan mountain without training in Nepal, and watched as her friend was attacked by a monkey in Indonesia.But interspersed in those slightly more crazy moments, Sue Bedfored and her friend "Sara the Stoic" experienced the sights, sounds, life, and culture of fifteen countries. Joined along the wa

In [97]:
for book in book_cards:
    {
    "UPC": "...",
    "Title": "...",
    "Price (£)": ...,
    "Rating": ...,
    "Genre": "...",
    "Availability": "...",
    "Description": "..."
}

books_data.append({
    "Title": title,
    "Price (£)": price_number,
    "Rating": rating
})

In [99]:
book_link = book.select_one("h3 a")["href"]

book_url = urljoin(catalogue_url, book_link)

detail_response = requests.get(
    book_url,
    headers=headers
)

detail_soup = BeautifulSoup(
    detail_response.content,
    "html.parser"
)

print(detail_soup.title.get_text())


    It's Only the Himalayas | Books to Scrape - Sandbox



In [101]:
book_record = {
    "UPC": upc,
    "Title": title,
    "Price (£)": price_number,
    "Rating": rating,
    "Genre": genre,
    "Availability": availability,
    "Description": description
}

books_data.append(book_record)
print(book_record)

{'UPC': 'a22124811bfa8350', 'Title': "It's Only the Himalayas", 'Price (£)': 45.17, 'Rating': 2, 'Genre': 'Travel', 'Availability': 'In stock (19 available)', 'Description': '“Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a few things her mother might classify as "stupid." She swam with great white sharks in South Africa, ran from lions in Zimbabwe, climbed a Himalayan mountain without training in Nepal, and wa “Wherever you go, whatever you do, just . . . don’t do anything stupid.” —My MotherDuring her yearlong adventure backpacking from South Africa to Singapore, S. Bedford definitely did a few things her mother might classify as "stupid." She swam with great white sharks in South Africa, ran from lions in Zimbabwe, climbed a Himalayan mountain without training in Nepal, and watched as her friend was attacked by a monkey in Indonesia.But interspersed

In [102]:
print(books_data[0])

{'Title': 'Set Me Free', 'Price (£)': 17.46, 'Rating': 5}


In [103]:
print(soup.find_all("li", class_="next"))

[<li class="next"><a href="catalogue/page-2.html">next</a></li>]


In [104]:
next_button = soup.select_one("li.next a")

print(next_button)

<a href="catalogue/page-2.html">next</a>


In [108]:
next_url = urljoin(catalogue_url, next_button["href"])
next_url

'https://books.toscrape.com/catalogue/page-2.html'